# Занятие 5. Детектирование объектов (YOLO)

## Цели занятия:
- Изучить архитектуру одностадийных детекторов (YOLO)
- Освоить библиотеку ultralytics в PyTorch
- Выполнить Fine-tuning модели на custom-датасете
- Оценить качество через mAP и визуализацию

---

## Шаг 1. Установка библиотеки

In [ ]:
# Install ultralytics package
# Uncomment if running for the first time:
# !pip install ultralytics

import torch
from ultralytics import YOLO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Шаг 2. Загрузка предобученной модели

In [ ]:
# Load pretrained YOLO model
# Using YOLOv8 nano for speed (smallest model)
model = YOLO('yolov8n.pt')  # This will download the model

print("Model loaded!")
print(f"Model type: YOLOv8n (nano)")

In [ ]:
# Run inference on sample image
# This will download a sample image from ultralytics
results = model('https://ultralytics.com/images/bus.jpg')

# Display results
results[0].show()

In [ ]:
# Inspect results
result = results[0]
print(f"Number of detections: {len(result.boxes)}")
print(f"\nDetected objects:")

for box in result.boxes:
    cls_id = int(box.cls[0])
    cls_name = result.names[cls_id]
    conf = box.conf[0].item()
    bbox = box.xyxy[0].tolist()
    print(f"  - {cls_name}: confidence={conf:.2f}, bbox={bbox}")

---
## Шаг 3. Подготовка датасета

Для fine-tuning нужен датасет в формате YOLO. 

**Структура папок:**
```
dataset/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

**Файл data.yaml:**
```yaml
path: ../datasets/custom
train: images/train
val: images/val
nc: 2
names: ['class1', 'class2']
```

In [ ]:
# For demonstration, we'll use a built-in dataset from ultralytics
# This downloads a small example dataset

# If you have a custom dataset, place it in ./datasets/custom/
# and create data.yaml file

import os
import yaml

# Check if custom dataset exists
custom_dataset_path = './datasets/custom/data.yaml'

if os.path.exists(custom_dataset_path):
    print("Custom dataset found!")
    with open(custom_dataset_path, 'r') as f:
        data_config = yaml.safe_load(f)
    print(f"Classes: {data_config['names']}")
    print(f"Number of classes: {data_config['nc']}")
    dataset_path = custom_dataset_path
else:
    print("No custom dataset found.")
    print("Using built-in COCO demo for inference demonstration.")
    dataset_path = None

---
## Шаг 4. Fine-tuning модели

In [ ]:
# Fine-tuning on custom dataset
# Only run if you have a custom dataset prepared

if dataset_path:
    print("="*50)
    print("Starting Fine-tuning...")
    print("="*50)
    
    # Train the model
    results = model.train(
        data=dataset_path,
        epochs=10,
        imgsz=640,
        batch=16,
        device=0 if torch.cuda.is_available() else 'cpu',
        project='runs/detect',
        name='custom_train'
    )
    
    print("\nFine-tuning completed!")
else:
    print("Skipping fine-tuning (no custom dataset).")
    print("\nTo fine-tune on your dataset:")
    print("1. Prepare dataset in YOLO format")
    print("2. Create data.yaml configuration file")
    print("3. Run model.train(data='path/to/data.yaml', epochs=10)")

---
## Шаг 5. Оценка и визуализация

In [ ]:
# Validate model performance
# If you have a trained model, load it:
# model = YOLO('runs/detect/custom_train/weights/best.pt')

# Run validation on pretrained model for demonstration
metrics = model.val()

print("\n" + "="*50)
print("VALIDATION METRICS")
print("="*50)
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# Predict on custom images
# You can replace this with your own image path

test_images = [
    'https://ultralytics.com/images/bus.jpg',
    'https://ultralytics.com/images/zidane.jpg'
]

for img_url in test_images:
    results = model(img_url)
    results[0].save(f'prediction_{img_url.split("/")[-1]}')
    results[0].show()

print("Predictions saved!")

---
## Шаг 6. Анализ результатов

In [ ]:
# Visualize training results (if fine-tuning was done)
import matplotlib.pyplot as plt
from PIL import Image
import os

# Check for training results
results_path = 'runs/detect/custom_train'

if os.path.exists(results_path):
    # Plot results.png if available
    results_img = os.path.join(results_path, 'results.png')
    if os.path.exists(results_img):
        img = Image.open(results_img)
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Training Results')
        plt.show()
else:
    print("No training results found.")
    print("Run fine-tuning first to see training curves.")

In [ ]:
# Export model to different formats
print("Model export options:")
print("- ONNX: model.export(format='onnx')")
print("- TensorRT: model.export(format='engine')")
print("- CoreML: model.export(format='coreml')")
print("- TFLite: model.export(format='tflite')")

---
## Домашнее задание

1. Дообучить модель на своём датасете (50-100 изображений)
2. Собрать мини-датасет и разметить его (через Roboflow или LabelImg)
3. Сравнить mAP предобученной модели и дообученной

---
## Контрольные вопросы

1. В чем разница между One-Stage и Two-Stage детекторами?
2. Что такое IoU и как он считается?
3. Зачем нужен Non-Maximum Suppression (NMS)?
4. Почему для Fine-tuning достаточно малого датасета?

In [ ]:
# Example: Creating a simple custom dataset structure
# Uncomment to create placeholder structure

"""
import os

# Create directory structure
for split in ['train', 'val']:
    os.makedirs(f'datasets/custom/images/{split}', exist_ok=True)
    os.makedirs(f'datasets/custom/labels/{split}', exist_ok=True)

# Create data.yaml
data_yaml = '''
path: ./datasets/custom
train: images/train
val: images/val
nc: 2
names: ['ball', 'box']
'''

with open('datasets/custom/data.yaml', 'w') as f:
    f.write(data_yaml)

print("Dataset structure created!")
print("Add your images and labels to the appropriate folders.")
"""